# Simulation Benchmark for Causal Effect Estimation

## Overview

This notebook demonstrates the simulation benchmark used to generate observational datasets with known ground-truth Average Treatment Effects (ATE).

The implementation of each data-generating process (DGP) lives in `src/simulation/`. This notebook focuses on **configuration, execution, and inspection of results** rather than repeating the underlying function definitions.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Make the project root importable whether this notebook is launched
# from the repository root or from notebooks/.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "simulated"
DATA_DIR.mkdir(parents=True, exist_ok=True)

from src.simulation.common import gen_covariate, gen_error, save_data
from src.simulation.engine1 import main_1
from src.simulation.engine2 import main_2
from src.simulation.engine3 import main_3
from src.simulation.engine4 import main_4
from src.simulation.engine5 import main_5
from src.simulation.engine6 import main_6
from src.simulation.engine7 import engine7_mcar, engine7_mar
from src.simulation.engine8 import main_8


## Shared simulation setup

The original project used \(p=200\) covariates and \(n=1000\) observations for the main benchmark examples. A fixed random seed makes the examples reproducible.


In [ ]:
rng = np.random.default_rng(123)

p = 200
n = 1000

w = rng.uniform(0, 1, size=(p, 1))
b = rng.uniform(0, 1, size=(p, 1))

X = gen_covariate(p, n, sigma=np.eye(p), rng=rng)
eps = gen_error(n, sd=1.0, rng=rng)

X.shape


## Engine 1 — Linear baseline with tunable confounding

This engine uses a logistic treatment-assignment model and a linear baseline outcome. The treatment effect is constant, so the ATE is analytical.


In [ ]:
a, k = 1, 1
for tau in [0, 0.5, 1]:
    D1 = main_1(X, w, a, k, b, tau, eps, rng=rng)
    save_data(D1, DATA_DIR / f"engine1_tau_{tau}.csv")

print("Engine 1 complete.")


## Engine 2 — Nonlinear outcome with constant treatment effect

The treatment model remains logistic, while the outcome uses nonlinear transformations and interactions among the first five covariates.


In [ ]:
tau_max = np.max(np.abs(X[4, :] ** 3))
for tau in [0, tau_max / 2, tau_max]:
    D2 = main_2(X, w, a, k, tau, eps, rng=rng)

print("Engine 2 complete.")


## Engine 3 — Interaction-heavy heterogeneous treatment effects

The treatment effect varies with covariates. The parameterization preserves a known population ATE under Gaussian covariates.


In [ ]:
v = np.random.default_rng(0).normal(size=p)
alpha, kappa, sd = 1, 1, 1

tau_list = [
    (0, 0, 0),
    (0, 1, 4),
    (0, 0, 4),
    (1, 1, 1),
    (2, 4, 6),
]

for tau0, tau1, tau2 in tau_list:
    D3 = main_3(p, n, v, alpha, kappa, tau0, tau1, tau2, sd, seed=0)

print("Engine 3 complete.")


## Engine 4 — Sparse confounding with heterogeneous treatment effects

Only selected covariates contribute to treatment assignment and the baseline outcome, while one selected covariate drives treatment-effect heterogeneity.


In [ ]:
SA = [0, 1, 2]
SY = [0, 3, 4]
j_star = 1

for t0, t1 in [(0, 0), (0, 1), (1, 1), (1, 4)]:
    D4 = main_4(X, SA, SY, w, a, k, b, t0, t1, j_star, eps, rng=rng)

print("Engine 4 complete.")


## Engine 5 — Overlap / positivity stress tests

Engine 5 modifies treatment assignment to create increasingly difficult overlap regimes while reusing outcome mechanisms from Engines 1–4.


In [ ]:
overlap_grid = [0.10, 0.05, 0.02, 0.01, 0.005]

for ep in overlap_grid:
    D5 = main_5(
        X, w, a, k, ep, 1.0, "Engine 1",
        eps=eps, b=b, rng=rng
    )

print("Engine 5 complete.")


## Engine 6 — Nonlinear treatment assignment and nonlinear outcome

This engine introduces nonlinear treatment assignment using \(X_1,\ldots,X_5\) and a nonlinear baseline outcome.


In [ ]:
D6, propensity6 = main_6(
    p=20, n=1000, a0=1, k=1, tau0=1, sd=1, seed=0
)

print("Engine 6 complete.")


## Engine 7 — Missingness stress tests

MCAR and MAR mechanisms are applied after data generation to evaluate robustness to missing outcomes.


In [ ]:
X6, A6, Y06, Y16, Y6 = D6
X6_nxp = X6.T

mcar_example = engine7_mcar(X6_nxp, A6, Y6, pi=0.8, seed=0)
mar_example = engine7_mar(
    X6_nxp,
    A6,
    Y6,
    gamma0=1.0,
    gammaA=0.25,
    gamma=np.zeros(X6_nxp.shape[1]),
    seed=0,
)

print("Engine 7 complete.")


## Engine 8 — Measurement error and heavy-tailed noise

The observed covariates contain measurement error and the outcome error follows a Gaussian mixture with a heavy-tailed component.


In [ ]:
for tau in [0, 2, 4, 6]:
    D8 = main_8(
        X, w, a, k, b, tau, sd=1,
        sigma_u=0.5,
        w_main=0.95,
        sd_multiplier=10.0,
        rng=rng,
    )

print("Engine 8 complete.")


## Data-to-text and LLM evaluation

Serialization, prompt construction, and evaluation metrics have also been moved out of the notebook:

- `src/data_to_text.py`
- `src/evaluation.py`

This keeps the notebook readable while preserving reusable project logic in Python modules.


In [ ]:
from src.data_to_text import preclean, small_to_text, large_to_subset, summary_stats, build_prompt
from src.evaluation import point_estimation, compute_uncertainty

example_path = DATA_DIR / "engine1_tau_1.csv"
clean_df = preclean(example_path)
clean_df.head()
